#### Creating Vector Embeddings for the knowledge-base to store in ChromaDB

- Step 1: Divide documents into Chunks

- Step 2: Convert chunks into vector embeddings

- Step 3: Visualize vector embeddings in 2D and 3D

In [1]:
# to install and upgrade
'''
%pip install -U langchain
%pip install -U langchain-chroma
%pip install -U langchain-huggingface
%pip install -U langchain-community
%pip install -U scikit-learn
%pip install -U plotly
%pip install -U transformers
%pip install -U sentence-transformers
%pip install -U nbformat
'''



'\n%pip install -U langchain\n%pip install -U langchain-chroma\n%pip install -U langchain-huggingface\n%pip install -U langchain-community\n%pip install -U scikit-learn\n%pip install -U plotly\n%pip install -U transformers\n%pip install -U sentence-transformers\n%pip install -U nbformat\n'

In [2]:
# imports

import os
import glob                                                                     # to find and list the files/folders paths
from transformers import AutoTokenizer                                          # to convert the input text into tokenIDs
from langchain_community.document_loaders import DirectoryLoader, TextLoader    # to load the directories and text inside the folders inside the directories
from langchain_text_splitters import RecursiveCharacterTextSplitter             # to convert the text into chunks
from langchain_huggingface import HuggingFaceEmbeddings                         # embedding model
from langchain_chroma import Chroma                                             # vector data store
from sklearn.manifold import TSNE                                               # to convert from higher dimensions to 2D or 3D
import plotly.graph_objects as go                                               # to visualize
import numpy as np

/var/folders/dv/bj8y54ys6czc90jxfd5b64gc0000gn/T/ipykernel_14808/640104295.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader    # to load the directories and text inside the folders inside the directories


In [3]:
# model and db

model = 'openai/gpt-oss-20b'

db_name = 'vector_db'


In [4]:
# to see the total number of files and total number of characters in the entire knowledge base

knowledge_base_path = "../knowledge-base/**/*.md"

files = glob.glob(knowledge_base_path, recursive=True)
print(f'Found {len(files)} files in knowledge base')

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += '\n\n'

print(f'Total characters in knowledge base: {len(entire_knowledge_base):,}')
print(f'Total words in the knowledge base: {len(entire_knowledge_base.split(' '))}')

Found 76 files in knowledge base
Total characters in knowledge base: 304,434
Total words in the knowledge base: 42187


In [5]:
# converting the text into tokens

tokenizer = AutoTokenizer.from_pretrained(model)
tokens = tokenizer.encode(entire_knowledge_base)
print(f'Total tokens for {model} is {len(tokens)}')

Total tokens for openai/gpt-oss-20b is 63555


- Satisfied the rule of thumb for tokens to word ratio i.e., for every 2/3 word, there will be 1 token

### Step 1 - Divinding documents into Chunks

#### Loading in everything into the knowledge base using LangChain's loaders

In [6]:
# we have a total of 76 .md files in our knowledge base and load each of the file into a list

folders = glob.glob("../knowledge-base/*")

documents = []

for folder in folders:
    doc_type = os.path.basename(folder)                                                                             # returns sub-folder names like company, contracts, employees and products
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding':'utf-8'})     # enter into each sub-folder and load all the files inside it that ends with .md extension
    folder_docs = loader.load()                                                                                     # all the files were loaded into folder_docs
    
    for doc in folder_docs:
       doc.metadata['doc_type'] = doc_type                                                                          # for each file, update the doc_type in metadata of the file as either [company, contracts, employees, products]
       documents.append(doc)                                                                                        # append all the modified documents

print(f'Loaded {len(documents)} documents')

Loaded 76 documents


In [7]:
# accessing a sample file via list

documents[0]

Document(metadata={'source': '../knowledge-base/products/Rellm.md', 'doc_type': 'products'}, page_content="# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intell

In [8]:
# divide the entire text into chunks using RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)              # text_splitter object with chunk size 1000 and overlap 200
chunks = text_splitter.split_documents(documents)

print(f'Divided into {len(chunks)} chunks\n')
print('------------------------------------')
print(f'First chunk: \n\n{chunks[0]}')

Divided into 413 chunks

------------------------------------
First chunk: 

page_content='# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.

## Features

### AI-Driven Analytics
Rellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.' metadata={'source'

In [9]:
# last chunk along with metadata (including the doc_type we have added)

print(f' Metadata of the last chunk: \n {chunks[412].metadata}')
print('----------------------------------')
print(f'Content in the last chunk: \n {chunks[412]}')

 Metadata of the last chunk: 
 {'source': '../knowledge-base/employees/Brandon Walker.md', 'doc_type': 'employees'}
----------------------------------
Content in the last chunk: 
 page_content='## Other HR Notes
- **Education:** Associate Degree in Information Technology from Phoenix Community College
- **Certifications:** CompTIA A+, working toward Network+ certification
- **Performance Improvement Plan:** Currently on 90-day PIP (started August 2023) focusing on response time improvements and customer communication skills
- **Skills:** Strong in system troubleshooting, SQL basics, and API debugging. Needs improvement in soft skills and time management.
- **Development:** Working with manager on structured troubleshooting approach and empathetic customer communication
- **Feedback:** Technically competent but struggles under pressure. Tends to focus on technical details rather than customer experience. Improving but needs consistent focus.' metadata={'source': '../knowledge-base/emplo

### Step 2 - Converting chunks into Vector Embeddings and store them in Chroma DB

In [10]:
# using open-source embedding model from HuggingFace [384D vectors]

embeddings = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2')                                     # embedding model

db_path = f'../{db_name}'                                                                               # db_path

if os.path.exists(db_path):                                                                             # check if the database exists in the current directory
    Chroma(persist_directory=db_path, embedding_function=embeddings).delete_collection()                # if db found, connect to the db and wipe out all the stored vectors and metadata

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_path)  # builds the actual database, (langchains vector store wrapper)
print(f'Vectorstore created with {vectorstore._collection.count()} documents')                          # Chroma's internal method `-collection.count()` to count the total records stored in the collection


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectorstore created with 413 documents


#### Description about Chroma

- In vector databases like Chroma, a `Collection` is the fundamental container used to store and organize related data

- Think of a Collection as the vector database equivalent of:
    - A Table in a Relational Database (like SQL).

    - A Collection in a Document Database (like MongoDB).

    - A Folder on your file system.

**What Does a Collection Store?**

- Inside a single Chroma collection, every entry contains four core parts:  

    1. **documents**: The raw text chunks (e.g., "LangChain is a framework...")

    2. **embeddings**: The numerical vector representations generated by your embedding model (e.g., [0.012, -0.453, 0.891, ...])

    3. **metadatas**: Optional key-value context attached to each chunk (e.g., {"author": "Alice", "page": 4})

    4. **ids**: A unique string identifier for each record (e.g., "id1") 

**How Collections Work in Practice**

- **Default Behavior**: If you do not specify a name, Chroma automatically uses a default collection named "langchain"

- **Separation of Scopes**: You create different collections to keep distinct datasets isolated. For instance: 

    - collection_a = "Company HR Policies"

    - collection_b = "Python Documentation"

**Embedding Model Rule**

- Every collection is tied to a specific vector dimension and embedding model. You should not mix vectors generated by different embedding models (e.g., OpenAI vs. Hugging Face) inside the same collection

In [11]:
# investigating the vectors

collection = vectorstore._collection                                                            # Chroma's underlying low level Python API from langchains higher level wrapper (vectorstore is the Langchain wrapper)
count = collection.count()

sample_embedding = collection.get(limit=1, include=['embeddings'])['embeddings'][0]
dimensions = len(sample_embedding)

print(f'There are {count:,} vectors with {dimensions:,} dimensions in the vector store')
print('\n-----------------------------------------------\n')
print(f'Sample Embeddings: \n{sample_embedding}')


''' 
`collection.get()`      - fetches stored records
`limit=1`               - restrict the output to 1 record
`include=['embeddings]` - instructs Chroma to return raw vector data (by default, Chroma omits full vector arrays to optimize speed)
`embeddings[0]`         - extract the first numerical vector list from the returned dictionary

'''

There are 413 vectors with 384 dimensions in the vector store

-----------------------------------------------

Sample Embeddings: 
[-7.94353932e-02  2.09135208e-02 -7.59418383e-02  5.89141324e-02
  3.71719338e-02  3.00868470e-02  7.36817271e-02  2.47455928e-02
 -2.21104920e-02 -2.67833751e-02 -1.69291589e-02  1.77901369e-02
  6.83952123e-02 -1.75439622e-02 -6.56956211e-02  3.76065485e-02
  3.86256650e-02 -6.62030000e-03 -5.92322797e-02 -4.20429632e-02
 -3.63235772e-02  3.90675180e-02 -5.29938191e-02 -4.47650254e-02
  1.81712657e-02  1.01755196e-02  3.97955030e-02  8.80592316e-02
 -2.72476897e-02 -4.93590869e-02  7.11433142e-02 -3.01459786e-02
  2.20190082e-02  1.87587049e-02 -6.19552769e-02  8.15793350e-02
 -1.12457849e-01 -3.32286060e-02 -3.82960886e-02 -4.32644039e-02
 -3.06527633e-02 -2.33631711e-02 -6.01168461e-02  7.91762210e-03
  6.32209182e-02  3.53679284e-02 -6.30453676e-02  2.71505248e-02
  4.78660800e-02  1.06016673e-01 -9.28903818e-02  2.18817522e-03
 -2.78206952e-02  1.155

" \n`collection.get()`      - fetches stored records\n`limit=1`               - restrict the output to 1 record\n`include=['embeddings]` - instructs Chroma to return raw vector data (by default, Chroma omits full vector arrays to optimize speed)\n`embeddings[0]`         - extract the first numerical vector list from the returned dictionary\n\n"

In [12]:
# accessing text from store instead of embeddings (by default, Chroma returns documents with embeddings as None)

sample_text_from_embeddings = collection.get(limit=1)
print(sample_text_from_embeddings)

{'ids': ['38551ad8-95d8-49c8-a558-292e643dc8c1'], 'embeddings': None, 'documents': ['# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.'], 'uris': Non

In [13]:
# explicitly asking for embeddings

print(collection.get(limit=1, include=['embeddings'])['embeddings'][0])

[-7.94353932e-02  2.09135208e-02 -7.59418383e-02  5.89141324e-02
  3.71719338e-02  3.00868470e-02  7.36817271e-02  2.47455928e-02
 -2.21104920e-02 -2.67833751e-02 -1.69291589e-02  1.77901369e-02
  6.83952123e-02 -1.75439622e-02 -6.56956211e-02  3.76065485e-02
  3.86256650e-02 -6.62030000e-03 -5.92322797e-02 -4.20429632e-02
 -3.63235772e-02  3.90675180e-02 -5.29938191e-02 -4.47650254e-02
  1.81712657e-02  1.01755196e-02  3.97955030e-02  8.80592316e-02
 -2.72476897e-02 -4.93590869e-02  7.11433142e-02 -3.01459786e-02
  2.20190082e-02  1.87587049e-02 -6.19552769e-02  8.15793350e-02
 -1.12457849e-01 -3.32286060e-02 -3.82960886e-02 -4.32644039e-02
 -3.06527633e-02 -2.33631711e-02 -6.01168461e-02  7.91762210e-03
  6.32209182e-02  3.53679284e-02 -6.30453676e-02  2.71505248e-02
  4.78660800e-02  1.06016673e-01 -9.28903818e-02  2.18817522e-03
 -2.78206952e-02  1.15563385e-02  7.48851942e-03  7.57691935e-02
 -7.94371031e-03 -3.52643542e-02 -5.56844622e-02 -2.63047125e-02
  2.07791440e-02 -2.71619

### Step 3 - Visualize

In [ ]:
# pre-work to visualize

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
print(doc_types)
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

''' 
doc_types = ['products', 'products', 'contracts', ........]

Lookup Index via .index(t)
If t = 'contracts',
then ['products', 'employees', 'contracts', 'company'].index('contracts') - returns position index as 2
then ['blue', 'green', 'red', 'orange'][2] - returns 'red' as the color for that doc_type
'''

['products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'products', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contracts', 'contr

" \ndoc_types = ['products', 'products', 'contracts', ........]\n\nLookup Index via .index(t)\nIf t = 'contracts',\nthen ['products', 'employees', 'contracts', 'company'].index('contracts') - returns position index as 2\nthen ['blue', 'green', 'red', 'orange'][2] - returns 'red' as the color for that doc_type\n"

In [ ]:
# reducing the dimensionality of the vectors to 2D using t-SNE
# t-SNE - t-distributed Stochastic Neighbor Embedding

tsne = TSNE(n_components=2, random_state=42)                                                
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    # scene={'xaxis_title':'x', 'yaxis_title':'y'}, is also same, because plotly API expects the input in dicts
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

"""
`n_components=2`            - indicates to reduce the high dimensionality vectors into 2D
`random_state=42`           - to make the output consistent over multiple runs, [Sets a seed for reproducible results every time the algorithm runs]
`fit_transform(vectors)`    - computes the 2D spatial co-ordinates for all the vector arrays,
                              reduced vectors are in shape (N,2), N-indicates total no. of chunks
                              reduced_vectors[:,0] - holds x_co-ordinates
                              reduced_vectors[:,1] - hollds y_co-ordinates for 2D
--------------------------------------------------
`go.Scatter()`              - Scatter plot
`mode='markers'`            - Displays the data points as individual dot markers (no connecting lines)
`marker = dict()`           - to style the marker for each data point with size, color and transparency
`text`                      - custom text to display when hovering over the datapoints instead of raw x,y co-ordinates
                              Type: {doc_type}
                              Text: documents[:100] -first hundred characters of the chunks
`hoverinfo='text'`          - configures plotly to display the formatted text when hovering
--------------------------------------------------
`update_layout`             - defines visual properties like title, width, height in pixels, trims white space margins around plot
                              like top=40, bottom=10, right=20, left=10
`fig.show()`                - renders the interactive HTML scatter plot

"""

"\n`n_components=2`            - indicates to reduce the high dimensionality vectors into 2D\n`random_state=42`           - to make the output consistent over multiple runs, [Sets a seed for reproducible results every time the algorithm runs]\n`fit_transform(vectors)`    - computes the 2D spatial co-ordinates for all the vector arrays,\n                              reduced vectors are in shape (N,2), N-indicates total no. of chunks\n                              reduced_vectors[:,0] - holds x_co-ordinates\n                              reduced_vectors[:,1] - hollds y_co-ordinates for 2D\n--------------------------------------------------\n`go.Scatter()`              - Scatter plot\n`mode='markers'`            - Displays the data points as individual dot markers (no connecting lines)\n`marker = dict()`           - to style the marker for each data point with size, color and transparency\n`text`                      - custom text to display when hovering over the datapoints instead of r

In [19]:
# reducing the dimensionality of the vectors using 3D

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# create 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x = reduced_vectors[:,0],
    y = reduced_vectors[:,1],
    z = reduced_vectors[:,2],
    mode = 'markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text = [f'Type: {t}<br>Text: {d[:100]}...' for t, d in zip(doc_types, documents)],
    hoverinfo = 'text'
)])

# defining visual properties
fig.update_layout(
    title = '3D Chroma Vector Store Visualization',
    scene = dict(xaxis_title = 'x', yaxis_title = 'y', zaxis_title = 'z'),
    width = 900,
    height = 700,
    margin = dict(r=10, b=10, l=10, t=40)
)

fig.show()